# STELLAR Overview

This tutorial explains the basic components of STELLAR. For illustration, it uses as system-under-test a simplified natural language intent recognition based on keyword counts.


## Architecture 
The simplified test generation pipeline of STELLAR is as follows:

```mermaid
flowchart LR
    A[Feature Configs<br/>content/style/perturbation] --> B[Test Input Generator<br/>prompting + optional RAG]
    B --> C[SUT<br/>LLM app under test]
    C --> D[Evaluators<br/>judge LLM + similarity metrics]
    D --> E[Search Algorithm<br/>rs / gs / nsga2 / astral]
    E --> B
    D --> F[Results + Dashboard<br/>failures, rates, heatmaps]

    classDef config fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#000;
    classDef generator fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#000;
    classDef sut fill:#FFF3E0,stroke:#EF6C00,stroke-width:2px,color:#000;
    classDef eval fill:#F3E5F5,stroke:#6A1B9A,stroke-width:2px,color:#000;
    classDef search fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#000;
    classDef results fill:#ECEFF1,stroke:#455A64,stroke-width:2px,color:#000;

    class A config;
    class B generator;
    class C sut;
    class D eval;
    class E search;
    class F results;
```
What each block does:

1. Feature Configs: define what kinds of inputs STELLAR is allowed to generate.
2. Test Input Generator: creates candidate prompts/questions from those feature definitions.
3. SUT: the system under test (for example, a chatbot or NLU-recognition system) executes the input.
4. Evaluators: scores the outputs from the SUT based on the inputs and decide whether a test case is a failure.
5. Search Algorithm: uses scores to decide which test cases to generate next (dynamic generation).
6. Results + Dashboard: stores artifacts and helps inspect failures interactively.


# Feature Configuration 

The feature configuration defines what kind of tests can be generated. The configuration is provided via a feature configuration file. A feature is classified into one of the types: 

| Type | Definition | Example | Usage | 
| --- | --- | --- | --- |
| Content | Defines the content of the utterance. | `intent` with the possible values e.g. INTENT_Navigation, INTENT_Climate | Use Case Specific |
| Style | Define the stylistic expression of the utterance. | `politeness`,  `formality`, `implicitness` | Generic | 
| Perturbation | Perturbations which are applied after a generated utterance. | `delete_words` or `introduce_fillers_static` | Generic |

The possible features are provided via the config file: [configs/features.config](./configs/features.json). 

To define custom feature types adopt the data model here: [models.py](./custom_models.py)

Each feature is *categorized* whether it is ordinal, categorical or continous for optimal handling in STELLAR.
A feature defines the ranges for its value and a numerical distribution for each value (null if equal).
Example:

```python
{
  "categorical_features": [
    {
      "name": "intent",
      "values": ["INTENT_Climate", "INTENT_Navigation"],
      "distribution": null
    },
    {
      "name": "word_perturbation",
      "values": ["none", "delete_words", "introduce_fillers_static", "introduce_homophones_static"],
      "distribution": null
    }
  ],
  "ordinal_features": [
    {
      "name": "politeness",
      "values": ["rude", "neutral", "polite"],
      "distribution": null
    },
    {
      "name": "slang",
      "values": ["formal", "neutral", "slangy"],
      "distribution": null
    },
    {
      "name": "implicitness",
      "values": ["explicit", "implicit"],
      "distribution": null
    },
    {
      "name": "verbosity",
      "values": ["short", "medium", "long"],
      "distribution": null
    }
  ]
}
```

## Utterance Generator

The utterance generator uses the feature configuration to generate test inputs.

To use a custom generator implement the ```generate_utterance``` function of ```UtteranceGenerator```.

Here: the utterance generator uses the __intent name__, __utterance examples__ and defined __features__ to generate an utterance.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# If notebook is in STELLAR/custom/, this resolves to STELLAR/
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
# ---- runtime setup (jupyter + env) ----
load_dotenv(find_dotenv(), override=False)

import json
import os
import random
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

from llm.features import FeatureHandler
from llm.llms import LLMType, pass_llm
from llm.model.models import Utterance
from llm.perturbations.apply_perturbations import apply_post_perturbations
from llm.utterance_generation.utterance_generator import UtteranceGenerator

from custom.custom_models import CustomContentInput, CustomStyleDescription

class CustomUtteranceGenerator(UtteranceGenerator):
    call_counter = 0
    
    # holds information of all available intents, with descriptions and examples
    INTENT_REFERENCE_PATH = repo_root / "custom" / "configs" / "intent_reference.json"

    def __init__(self, feature_handler: Optional[FeatureHandler] = None):
        super().__init__(feature_handler=feature_handler)
        self.intent_reference = self._load_intent_reference()

    @classmethod
    def _load_intent_reference(cls) -> Dict[str, Any]:
        if cls.INTENT_REFERENCE_PATH.exists():
            try:
                return json.loads(cls.INTENT_REFERENCE_PATH.read_text(encoding="utf-8"))
            except Exception:
                pass
        # fallback when no descriptions file exists
        return {
            "intents": {
                "INTENT_Climate": {
                    "description": "User asks to change cabin climate settings.",
                    "samples": [
                        "Turn on the AC.",
                        "Set temperature to 20 degrees.",
                        "It is too warm in here.",
                    ],
                },
                "INTENT_Navigation": {
                    "description": "User asks for route guidance or destination help.",
                    "samples": [
                        "Navigate to the station.",
                        "Show directions to the airport.",
                        "I need help getting somewhere.",
                    ],
                },
            },
            "slots": {},
        }

    @staticmethod
    def _has_llm_credentials() -> bool:
        env_names = (
            "OPENAI_API_KEY"
        )
        return any(os.getenv(name) for name in env_names)

    @staticmethod
    def _coerce_llm_type(llm_type: str | LLMType) -> LLMType:
        return llm_type if isinstance(llm_type, LLMType) else LLMType(llm_type)

    def _style_prompt(self, feature_values: Dict[str, Any]) -> str:
        style = CustomStyleDescription.model_validate(feature_values)
        result = style.model_dump_json(
            exclude={"word_perturbation", "char_perturbation"},
            indent=2,
        )
        if "num_words" in feature_values:
            result = f'{{"num_words": {feature_values["num_words"]}}}\n' + result
        return "Style features:\n" + result

    def _content_prompt(self, content_input: CustomContentInput) -> str:
        intents = self.intent_reference.get("intents", {})
        info = intents.get(content_input.intent, {"description": "Unknown intent", "samples": []})

        samples = info.get("samples", [])[:3]
        slots = self.intent_reference.get("slots", {})
        
        # provide few shot examples for the intent
        sample_lines = "\n".join(f"- {s}" for s in samples) if samples else "- No examples"

        return f"""INTENT:
            {content_input.intent}
            
            SLOTS:
            {slots}
            
            EXAMPLES:
            {sample_lines}
            
            RULE:
            Return exactly one natural user utterance for this intent.
            Do not use placeholders.
            Keep it concise (prefer <= 15 words unless style requires longer).
            """

    @staticmethod
    def _perturbation_prompt(feature_values: Dict[str, Any]) -> str:
        word_p = feature_values.get("word_perturbation", "none")
        if word_p == "introduce_fillers_llm_combined":
            return (
                "At the end, insert 1-2 natural filler words. "
                "Return only the modified text."
            )
        if word_p == "introduce_homophones_llm_combined":
            return (
                "At the end, replace 1-2 words with valid homophones if possible. "
                "Return only the modified text."
            )
        return "No extra LLM-side perturbation instruction."

    @staticmethod
    def _apply_final_style_rules(question: str, style: CustomStyleDescription) -> str:
        question = question.strip(" \t\n.,!?")
        if style.slang == "slangy" and not question.lower().startswith("hey"):
            question = f"Hey, {question}"
        if style.politeness == "polite" and not question.lower().endswith("please"):
            question = f"{question}, please"
        return question[:1].upper() + question[1:] if question else question

    @staticmethod
    def _fallback_question(content_input: CustomContentInput, style: CustomStyleDescription) -> str:
        """ Customize based on your use case. """

        fallback = {
            "INTENT_Climate": {
                "explicit": ["turn on the AC", "set the temperature to 20 degrees", "start the fan", "turn on the heating"],
                "implicit": ["it is too warm in here", "the cabin feels cold"],
            },
            "INTENT_Navigation": {
                "explicit": ["navigate to the station", "show directions to the airport", "find a route home", "open the map"],
                "implicit": ["I need help getting somewhere", "help me reach my destination"],
            },
        }
        text = random.choice(fallback[content_input.intent][style.implicitness])
        if style.verbosity == "medium":
            text = f"{text} right now"
        elif style.verbosity == "long":
            text = f"{text} when you have a moment"
        return text

    def build_prompt(self, content_input: CustomContentInput, feature_values: Dict[str, Any]) -> str:
        """ Customize based on your use case. """
        
        style_block = self._style_prompt(feature_values)
        content_block = self._content_prompt(content_input)
        perturb_block = self._perturbation_prompt(feature_values)

        return f"""You generate human-like in-car assistant utterances.

                {style_block}
                
                {content_block}
                
                Perturbations:
                {perturb_block}
                
                Guidelines:
                - Preserve intent semantics.
                - Apply style naturally.
                - Return only one utterance.
                """

    def generate_utterance(
        self,
        seed: Optional[str],
        ordinal_vars: List[float],
        categorical_vars: List[int],
        llm_type: str | LLMType,
    ) -> Utterance:
        feature_values = self.feature_handler.get_feature_values_dict(
            ordinal_feature_scores=ordinal_vars,
            categorical_feature_indices=categorical_vars,
        )

        content_input = CustomContentInput(intent=feature_values["intent"])
        style = CustomStyleDescription.model_validate(feature_values)
        selected_llm = self._coerce_llm_type(llm_type)

        text = ""
        if selected_llm != LLMType.MOCK and self._has_llm_credentials():
            prompt = self.build_prompt(content_input, feature_values)
            for _ in range(5):
                try:
                    result = pass_llm(
                        msg=prompt,
                        system_message="You are a concise utterance generator.",
                        llm_type=selected_llm,
                        temperature=0.6,
                    )
                    if result:
                        text = str(result).strip()
                        if text:
                            break
                except Exception:
                    continue

        if not text:
            text = self._fallback_question(content_input, style)
            text = self._apply_final_style_rules(text, style)

        text = apply_post_perturbations(text, feature_values)
        self.call_counter += 1

        return Utterance(
            question=text,
            seed=seed,
            ordinal_vars=ordinal_vars,
            categorical_vars=categorical_vars,
            content_input=content_input,
        )


if __name__ == "__main__":
    features_path = repo_root / "custom" / "configs" / "features.json"
    handler = FeatureHandler.from_json(str(features_path))
    gen = CustomUtteranceGenerator(handler)
    sample = handler.sample_feature_scores()

    u_mock = gen.generate_utterance(None, sample.ordinal, sample.categorical, LLMType.MOCK)
    print("MOCK:")
    print(u_mock.model_dump_json(indent=2))

    u_llm = gen.generate_utterance(None, sample.ordinal, sample.categorical, LLMType.GPT_4O_MINI)
    print("LLM:")
    print(u_llm.model_dump_json(indent=2))

## System Under Test

The system under test receives the generated textual test input and produces an output. The system is implemented by implementing the ```simulate``` method from the class ```Simulator```. In the example below, the system receives a list of utterances and generates a list of outputs (batch processing).

__Example Test Input__: "Activate the climate."


__System Response__:

```json
{
  "intent": "INTENT_Climate",
  "probabilities": {
    "INTENT_Climate": 0.7,
    "INTENT_Navigation": 0.3
  },
  "score": 0.7
}

In [ ]:
"""The system under test."""
from __future__ import annotations
import re
from typing import List

from opensbt.simulation.simulator import Simulator
from llm.model.qa_simout import QASimulationOutput
from llm.model.models import Utterance

from custom.custom_models import CustomOutputModel

class CustomSUT(Simulator):
    """Classifies utterances' intents by keywords, with an intentional navigation bias."""

    ipa_name = "custom_keyword_intent_classifier"

    @staticmethod
    def _predict(text_input: str) -> CustomOutputModel:
        """This method is implementation specific. Here: Returns probabilities from keyword counts to mock NLU.
           Implement this function to pass the text_input to your SUT.
        """

        CLIMATE_KEYWORDS = {"ac", "air", "conditioning", "climate", "fan", "heat", "heating", "cold", "temperature"}
        NAVIGATION_KEYWORDS = {"navigate", "navigation", "route", "directions", "destination", "map", "drive"}

        tokens = re.findall(r"[a-z]+", text_input.lower())
        climate_hits = sum(token in CLIMATE_KEYWORDS for token in tokens)
        navigation_hits = sum(token in NAVIGATION_KEYWORDS for token in tokens)
        total_hits = climate_hits + navigation_hits

        if total_hits == 0:
            probabilities = {"INTENT_Climate": 0.5, "INTENT_Navigation": 0.5}
        else:
            probabilities = {
                "INTENT_Climate": climate_hits / total_hits,
                "INTENT_Navigation": navigation_hits / total_hits,
            }

        intent = max(probabilities, key=probabilities.get)  # This will now be "INTENT_Climate" or "INTENT_Navigation"
        return CustomOutputModel(
            intent=intent,
            probabilities=probabilities,
            score=probabilities[intent],
        )

    @staticmethod
    def simulate(
        list_individuals: List[List[Utterance]],
        variable_names: List[str],
        scenario_path: str,
        sim_time: float,
        time_step: float = 10,
        do_visualize: bool = False,
        temperature: float = 0,
        context: object = None,
        llm_type=None,
        **kwargs,
    ) -> List[QASimulationOutput]:
        results: List[QASimulationOutput] = []
        """ Implement this function to run the execution for a given list of utterances and return the results. """

        for utterance_group in list_individuals:
            utterance = utterance_group[0]

            # Run the classifier; modify this call or underlying function based on your use case
            output = CustomSUT._predict(utterance.question)

            # The framework collects the `answer` and the `raw_output`.
            utterance.answer = output.intent
            utterance.raw_output = output.model_dump()

            # For each utterance a QASimulationOutput instance is generated
            results.append(
                QASimulationOutput(
                    utterance=utterance,
                    model="None",
                    ipa=CustomSUT.ipa_name
                )
            )

        return results

if __name__ == "__main__":
    import json
    utterance = Utterance(
        question="Turn on the AC please",
        seed=None,
        ordinal_vars=[0.2, 0.8],
        categorical_vars=[1, 0],
        content_input=CustomContentInput(intent="INTENT_Climate"),
    )

    print("Utterance:")
    print(utterance.model_dump_json(indent=2))

    simout = CustomSUT.simulate(
        list_individuals=[[utterance]],   # one individual with one utterance
        variable_names=["utterance"],
        scenario_path=".",
        sim_time=1.0,
    )
    print("Response:")
    print(json.dumps(simout[0].utterance.raw_output, indent=2))

## Fitness Function

The fitness function evaluates an executed test and guides the generation of new tests.

In [ ]:
from typing import Tuple

from opensbt.evaluation.fitness import Fitness
from llm.model.qa_simout import QASimulationOutput


"""This Fitness function uses the probability of the expected intent to be minimized to find failures.
"""
class CustomFitness(Fitness):

    @property
    def min_or_max(self):
        """Return 'min' if the fitness should be minimized, 'max' if it should be maximized.
           For each objective assign one direction.
        """
        return ("min",)

    @property
    def name(self):
        """Name of the objective. You can add multiple objectives by returning a tuple of names."""
        return ("probability_expected_intent",)

    def eval(self, simout: QASimulationOutput, **kwargs) -> Tuple[float]:
        """ This fitness function received the execution output object and
            return the intent probability of the expected intent.
        """
        if simout is None or simout.utterance is None:
            return (1.0,)

        # retrieve raw output
        raw_output = simout.utterance.raw_output or {}

        # retrieve content input
        content_input = simout.utterance.content_input

        expected_intent = getattr( content_input, "intent", None)
        probabilities = raw_output.get("probabilities", {})
        
        if expected_intent not in probabilities:
            return (1.0,)

        # assign score (take already provided probability)
        score = float(probabilities[expected_intent])
        
        raw_output["_fitness_debug"] = {
            "expected_intent": expected_intent,
            "predicted_intent": raw_output.get("intent"),
            "probabilities": probabilities,
        }

        return (score,)
        
# 4) Evaluate fitness for one simulation output (batch processing in code)
fitness = CustomFitness()
fitness_value = fitness.eval(simout[0])[0]
print("System Output:")
print(json.dumps(simout[0].utterance.raw_output, indent=2))
print("Fitness score:", fitness_value)

## Oracle

The oracle decides if a test is failure or not. It returns 0 if a test fails, otherwise 1.

This oracle is using a threshold on the fitness function result of ```CustomFitness```.

Here a test is a failure if the probability of the classified intent is below 0.5.


In [ ]:
from llm.eval.critical import CriticalMerged, CriticalByFitnessThreshold
import numpy as np

fitness = CustomFitness()

critical = CriticalMerged(
    fitness_names=fitness.name,
    criticals=[
        (
            CriticalByFitnessThreshold(mode="<", score=0.5),
            ["probability_expected_intent"],
        ),
    ],
    mode="or",
)

vector_fitness = np.array([fitness_value], dtype=float)
is_critical = critical.eval(vector_fitness, simout=simout)

print("Fitness score:", fitness_value)
print("Is critical:", is_critical)

## Search Algorithm

STELLAR integrates multiple algorithms for dynamic test generation.

| Goal | Name | Flag | Explanation | 
|---|---|---|---|
| Explorative testing | Random Search | `rs` | Randomized testing for basic dataset generation | 
| Failure-oriented testing | NSGA-II | `nsga2` | Reuses feedback of past executions to focus on promising test cases |
| Combinatorial testing for feature coverage | T-wise | `gs` | Targets combinatorial interactions systematically |
| Safety-oriented combinatorial testing | ASTRAL | `astral` | Combinatorial safety testing |


## Experiments Setup

Combine all the previously defined components to build your experiment.

In [ ]:
from __future__ import annotations
############# Resolve path to config file
from pathlib import Path

# Resolve repo root (folder that contains both "custom" and "llm")
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (
    (repo_root / "custom").exists() and (repo_root / "llm").exists()
):
    repo_root = repo_root.parent

features_path = repo_root / "custom" / "configs" / "features.json"
print("Using features config:", features_path)
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if p.name == "STELLAR")
for child in repo_root.iterdir():
    if child.is_dir():
        sys.path.insert(0, str(child))
##################

from llm.features import FeatureHandler

import os

from opensbt.algorithm.nsga2_optimizer import NsgaIIOptimizer

from llm.eval.critical import CriticalByFitnessThreshold, CriticalMerged
from llm.model.qa_problem import QAProblem

from custom.custom_fitness import CustomFitness
from custom.custom_generator import CustomUtteranceGenerator
from custom.custom_sut import CustomSUT
from custom.presets import Hyperparameters
from custom.run_setup import build_search_config

# 1) Set hyperparameters
hp = Hyperparameters(
    algorithm="nsga2",
    population_size=5,
    n_generations=1000,
    max_time="00:00:15",
    seed=1,
    generator_llm="mock",  # change to e.g., gpt-4o-mini for LLM-based generation
    wandb_mode="disabled",
    features_config = features_path
)

# 2) Build search config
config = build_search_config(hp)

# 3) Build fitness and critical function
fitness = CustomFitness()

critical = CriticalMerged(
    fitness_names=fitness.name,
    criticals=[
        (
            CriticalByFitnessThreshold(mode="<", score=0.5),
            ["probability_expected_intent"],
        ),
    ],
    mode="or",
)

# 4) Build the QAProblem directly
problem_name = (
    f"CUSTOM_{hp.algorithm}_p{config.population_size}"
    f"_g{config.n_generations}_t-{hp.max_time}_s{hp.seed}"
).replace(":", "-")

problem = QAProblem(
    problem_name=problem_name,
    scenario_path=os.getcwd(),
    xl=[0],
    xu=[1],
    simulation_variables=["utterance"],
    fitness_function=fitness,
    critical_function=critical,
    simulate_function=CustomSUT.simulate,  # connection to your SUT
    seed_utterances=["turn on the AC"],
    context={},
    seed=hp.seed,
    names_dim_utterance=["utterance"],
    feature_handler_config_path=hp.features_config,
    question_generator=CustomUtteranceGenerator(),
)

# 5) Build the optimizer
optimizer = NsgaIIOptimizer(problem=problem, config=config)

## Run your experiment from code

Run the built experiment.

In [ ]:
# Run the search
result = optimizer.run()

# Save artifacts to the optimizer output folder
result.write_results(
    results_folder=optimizer.save_folder,
    params=optimizer.parameters,
    search_config=config,
)

print("Done.")
print("Results folder:", optimizer.save_folder)
print("Execution time (s):", f"{result.exec_time:.2f}")

## Run your experiment from CLI

Run the test generation from the project root. Change the ```generator_llm``` to an LLM, e.g., gpt-4o-mini for LLM-based generation.


In [ ]:
!python -m custom.main \
  --preset test \
  --algorithm nsga2 \
  --population_size 5 \
  --n_generations 1000 \
  --max_time 00:00:15 \
  --seed 1 \
  --generator_llm mock \
  --wandb_mode disabled

### Supported Parameters

| Name | Meaning | Example |
|---|---|---|
| `algorithm` | `nsga2` (guided evolutionary search), `random` (randomized search), `t-wise`, `astral` | `nsga2` |
| `population_size` | Test inputs kept per generation. For `random`, the total number of samples. | `10` |
| `n_generations` | Number of search iterations. | `10` |
| `max_time` | Wall-clock budget `HH:MM:SS`; the search stops at the first limit reached. | `00:10:00` |
| `seed` | Random seed for reproducibility. | `1` |
| `generator_llm` | Model used by the generator. `mock` keeps everything offline; use e.g. `gpt-4o-mini` for an LLM generator. | `mock` |
| `wandb_mode` | Experiment tracking mode: `disabled`, `offline`, or `online`. | `disabled` |
| `wandb_project` | Optional Weights & Biases project name. | `stellar-custom` |
| `wandb_entity` | Optional Weights & Biases organization or user that owns the project. | `my-organization` |
| `features_config` | Path to the feature-space JSON. | `custom/configs/features.json` |
| `results_folder` | Output directory (`None` → framework default `./results`). | `None` |

# Tracking in Weights-and-Biases (W&B)

[W&B](https://wandb.ai/site/) is a tracking service for experiments. W&B is disabled by default. 
For local tracking use `--wandb_mode offline`.

Specify the project via `-wandb_project stellar-custom`. 

To connect with your W&B account use: `wandb login` or set `WANDB_API_KEY`.

Then use `--wandb_mode online` and `--wandb_project stellar-custom`.

# Analyse Results

STELLAR generates multiple result artefacts in the ```result``` folder in the root execution path for system analysis and debugging.

All test case are stored in ```all_utterances.json```.

In [ ]:
import json
import matplotlib.pyplot as plt

from pathlib import Path
import os

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "custom").is_dir() and (path / "llm").is_dir()
)
os.chdir(project_root)

sample_root = project_root / "custom" / "result_samples"
time_budget_seconds = 15

print("********* Failure Counts ************")

fig, axis = plt.subplots(figsize=(9, 5))
for algorithm in ("random", "nsga2"):
    sample_path = sample_root / algorithm / "all_utterances.json"
    entries = json.loads(sample_path.read_text(encoding="utf-8"))
    cumulative_failures = []
    failures = 0
    for entry in entries:
        failures += int(entry.get("is_critical", False))
        cumulative_failures.append(failures)

    elapsed_seconds = [
        time_budget_seconds * (index + 1) / len(entries)
        for index in range(len(entries))
    ]
    axis.step(elapsed_seconds, cumulative_failures, where="post", label=algorithm)
    print(f"{algorithm}: {failures} failures found across {len(entries)} generated tests")

axis.set_xlabel("Estimated elapsed time (seconds)")
axis.set_ylabel("Cumulative failures")
axis.set_xlim(0, time_budget_seconds)
axis.set_ylim(bottom=0)
axis.grid(axis="y", alpha=0.3)
axis.legend(title="Algorithm")
plt.show()